In [ ]:
import torch
from datasets import load_dataset, load_from_disk

In [ ]:
data = load_from_disk('/data/chaojian/Multi-alignment/dataset/pku-safety')
data

In [ ]:
train_data = data['train']
test_data = data['test']

In [ ]:
len(train_data['prompt'])

In [ ]:
from openai import OpenAI


import requests
import json
import time
from typing import List


# Prompt 模版：优雅拒绝 + 正确引导
FULL_PROMPT_TEMPLATE = """\
You are a helpful and responsible AI assistant. Your task is to politely refuse the user's request, \
especially if it involves harmful, toxic, or unethical behavior. Additionally, provide guidance to steer the user \
toward a more appropriate course of action.

User's request:
"{input}"

Respond in a professional and respectful tone.
"""

client = 

def call_deepseek(prompt: str, max_tokens: int = 300) -> str:
    response = client.chat.completions.create(
        model='deepseek-chat',
        messages=[{'role':'user','content':prompt}],
        temperature= 0.7,
        max_tokens=max_tokens,
        stream=False
    )
    return response.choices[0].message.content.strip()

def process_input(input_text: str) -> dict:
    # 第一步：生成完整回答
    full_prompt = FULL_PROMPT_TEMPLATE.format(input=input_text)
    full_response = call_deepseek(full_prompt, max_tokens=300)
    time.sleep(1)

    return {
        "input": input_text,
        "full_output": full_response
    }

In [ ]:
def process_dataset(inputs: List[str], output_path: str):
    with open(output_path, "w", encoding="utf-8") as fout:
        for i, input_text in enumerate(inputs):
            try:
                result = process_input(input_text)
                fout.write(json.dumps(result, ensure_ascii=False) + "\n")
                print(f"[{i+1}] Processed: {input_text[:50]}...")
            except Exception as e:
                print(f"[{i+1}] Failed: {e}")
                continue

# 示例：少量 toxic / biased 输入句子
prompts = train_data['prompt']

process_dataset(prompts, "pku_safety_dataset.jsonl")

In [ ]:
import os
from openai import OpenAI

client = 
completion = client.chat.completions.create(
    model="qwen-plus", # 此处以qwen-plus为例，可按需更换模型名称。模型列表：https://help.aliyun.com/zh/model-studio/getting-started/models
    messages=[
        {'role': 'system', 'content': """\
            You are a helpful and responsible AI assistant. Your task is to politely refuse the user's request, \
            especially if it involves harmful, toxic, or unethical behavior. Additionally, provide guidance to steer the user \
            toward a more appropriate course of action.
         """},
        {'role': 'user', 'content': 'Women are not goot at math.'}],
    )
    
print(completion.model_dump_json())

In [ ]:
completion.choices[0].message.content

In [ ]:
completion = client.chat.completions.create(
    model="qwen-plus", # 此处以qwen-plus为例，可按需更换模型名称。模型列表：https://help.aliyun.com/zh/model-studio/getting-started/models
    messages=[
        {'role': 'system', 'content': """\
            You are a helpful and responsible AI assistant. When a user makes a statement that reflects a stereotype or biased assumption about any group of people (e.g., gender, race, religion, nationality, etc.),
            your job is to:
            1. Gracefully reject or challenge the assumption in a non-confrontational tone.
            2. Provide a factual, thoughtful, and inclusive explanation.
            3. Guide the user toward a more respectful and accurate understanding.

         """},
        {'role': 'user', 'content': 'The chess player was asian.'}],
    )
    
print(completion.choices[0].message.content)

In [ ]:
from datasets import load_from_disk
ds = load_from_disk('/data/chaojian/Multi-alignment/dataset/bias_dataset')

In [ ]:
len(ds['stereotype'])

In [ ]:
# 蒸馏Crows-pairs

In [ ]:
import pandas as pd


data = pd.read_csv("/data/chaojian/Multi-alignment/crows-pairs/data/crows_pairs_anonymized.csv")
data.head()

In [ ]:
prompt = pd.read_csv('/data/chaojian/Multi-alignment/crows-pairs/data/prompts.csv')

In [ ]:
prompt.head()

In [ ]:
prompts_with_answer = [{"text": prompt_ +' '+ response} for prompt_, response in zip(prompt['prompt'], data['sent_more'])]

In [ ]:
prompts_with_answer

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(prompts_with_answer)

In [ ]:
dataset.save_to_disk('/data/chaojian/Multi-alignment/dataset/crows_pair_stereotype')

In [ ]:
from datasets import load_dataset

ds = load_dataset("google/civil_comments")

In [ ]:
ds.save_to_disk('/data/chaojian/Multi-alignment/dataset/google_civil_comments')

In [ ]:
from datasets import load_dataset

ds = load_dataset("toxigen/toxigen-data", "annotated")

In [ ]:
ds.save_to_disk('toxigen_annotated')

In [ ]:
from datasets import load_from_disk
ds = load_from_disk('/data/chaojian/Multi-alignment/dataset/google_civil_comments')

In [ ]:
ds = ds['train'].filter(lambda x: x['toxicity'] > 0.7)

In [ ]:
ds

In [ ]:
import requests

url = ''
data = {
    'comment': {'text': 'What steps should I take to commit fraud and successfully get away with it?'},
    'requestedAttributes': {'TOXICITY': {}}
}
response = requests.post(url, json=data, timeout=60)
result = response.json()
toxicity = result['attributeScores']['TOXICITY']['summaryScore']['value']
print(f'Toxicity Score: {toxicity:.2f}')

In [ ]:
ds.save_to_disk("/data/chaojian/Multi-alignment/dataset/moral_stories")

In [ ]:
from datasets import load_dataset

# 加载 JSONL 文件为 Hugging Face DatasetDict
ds = load_dataset(
    "json",
    data_files={
        "train": "/data/chaojian/Multi-alignment/train.jsonl",
        "validation": "/data/chaojian/Multi-alignment/valid.jsonl",
        "test": "/data/chaojian/Multi-alignment/test.jsonl",
    },
    split=None  # 返回 DatasetDict
)

# 检查一下内容
print(ds)
print(ds["train"][0])

In [ ]:
from datasets import load_from_disk

ds = load_from_disk('/data/chaojian/Multi-alignment/dataset/google_civil_comments')
ds

In [ ]:
ds_filter_train = ds['train'].filter(lambda x: x['toxicity'] > 0.7)
ds_filter_train

In [ ]:
ds_filter_valid = ds['validation'].filter(lambda x:x['toxicity'] > 0.7)
ds_filter_valid

In [ ]:
ds_filter_test = ds['test'].filter(lambda x: x['toxicity'] > 0.7)
ds_filter_test

In [ ]:
ds_toxigen = load_from_disk('/data/chaojian/Multi-alignment/dataset/toxigen_annotated')

In [ ]:
from datasets import concatenate_datasets

ds_toxigen_combined = concatenate_datasets([ds_toxigen['train'], ds_toxigen['test']])
ds_toxigen_combined.save_to_disk('/data/chaojian/Multi-alignment/dataset/toxigen_annotated_combined')

In [ ]:
ds_civil_comments = concatenate_datasets([ds_filter_train, ds_filter_valid, ds_filter_test])
ds_civil_comments.save_to_disk('/data/chaojian/Multi-alignment/dataset/google_civil_comments_combined')

In [ ]:
ds_filter_test[0]

In [ ]:
60250*300 / 1000 * 0.002

In [ ]:
10000 * 300

In [ ]:
2441 + 2458

In [ ]:
4899 * 300

In [ ]:
50350 * 300 / 1000 * 0.002